# So how does the generation work?

Here, we are using the openAI API to send queries and obtain LLM (GPT 3.5) responses from openAI. 

We use the langchain class AzureOpenAI to manage this interaction. Then the process is extremely simple: 
- define an input prompt
- use methods from the AzureOpenAI class to generate text based on the input prompt
- display the reponse

Be mindful that every LLM has a limited context length that limites the maxmimum input and output it can handle. Thus, if the output it generates becomes too long, the answer will be cut off (which is annyoing). This is a general problem that can be addressed in many ways - the simplest one is trying to use prompt engineering to prevent (most of) these cases from happening.

In [1]:
import os # for environment variables
from langchain_openai import ChatOpenAI, AzureChatOpenAI # get the AzureOpenAI class

from IPython.display import Markdown, display # for displaying markdown

In [2]:
def get_chat_generator(model_name: str) -> AzureChatOpenAI:
    if model_name == "gpt-35-turbo-instruct":
        return ChatOpenAI(
            model_name="gpt-3.5-turbo",
            temperature=0,
            openai_api_key=os.getenv("OPENAI_API_KEY"),
        )

    elif model_name == "gpt-4.1-mini":
        return AzureChatOpenAI(
            model_name="gpt-4.1-mini",
            temperature=0,
            azure_endpoint=os.getenv("GPT_41_MINI_ENDPOINT"),
            api_key=os.getenv("GPT_41_MINI_API_KEY"),
            api_version="2024-12-01-preview",
        )

    else:
        raise ValueError(f"{model_name} is not a valid chat generator name")

In [3]:
input_prompt = "Write someting."

# input_prompt = "How are you?"

# input_prompt = "How's tricks mate?"

# input_prompt = "Provide a short history of the football club FC Bayern Munich. Focus on its glory and success. Do not use more than 100 words."

In [4]:
chat_generator = get_chat_generator("gpt-35-turbo-instruct")

In [5]:
response = chat_generator.invoke(input_prompt)
display(Markdown(response.content))

Sure! Here is a short poem for you:

In the quiet of the night,
Stars twinkle in the sky so bright.
A gentle breeze whispers through the trees,
As I lay here, feeling at ease.

The world is still, the world is calm,
And I find peace in this soothing balm.
I close my eyes and drift away,
Into dreams where I can play.

So let the night embrace me tight,
And guide me through until first light.
For in these moments, I find my rest,
And know that I am truly blessed.

Note that there is no retrieval augmented generation yet:

In [6]:
input_prompt = "Explain the role of a trust council at the Alexander Thamm GmbH. If you don't know the answer, say that you don't know."
response = chat_generator.invoke(input_prompt)
display(Markdown(response.content))

I don't have specific information about the role of a trust council at Alexander Thamm GmbH.

# Tasks

### Task 1.1: If everything in the cell above works, the model should tell you that it doesn't know. Do you know why?

Because it doesn't have access to any of our internal documents on the Trust Council at [at]. Since we ware telling it not go give us information if it doesn't know, we prevent it from generating any answer that may seem reasonable but is unfounded.

### Task 1.2: How can you change the prompt to get the model to hallucinate? What other ways can you think of to prevent or indicate the occurence of an hallucination in an answer?

A generative model is trained to generate stuff. So it will produce text that sounds more or less meaningful if you ask it to. The code below will produce text on the trust council at [at] that sounds reasonable, but you can notice at some points that it makes up stuff and doesn't know about the specific of a trust council at [at].

In [7]:
input_prompt = "Explain the role of a trust council at the Alexander Thamm GmbH. Be brief."
response = chat_generator.invoke(input_prompt)
display(Markdown(response.content))

The trust council at Alexander Thamm GmbH is responsible for overseeing and managing the company's trust funds and assets. They ensure that the funds are invested wisely and in accordance with the company's goals and objectives. The trust council also provides guidance and advice on financial matters to help the company achieve its long-term financial stability and growth.

It is also often useful to ask the model to quantify it's uncertainty.

In [8]:
input_prompt = "Explain the role of a trust council at the Alexander Thamm GmbH. Be brief. Quantify your uncertainty in this reponse."
response = chat_generator.invoke(input_prompt)
display(Markdown(response.content))

The role of a trust council at Alexander Thamm GmbH is to oversee and manage the company's trust funds and assets, ensuring they are used in accordance with the trust's objectives and beneficiaries' interests. They make decisions on investments, distributions, and other financial matters related to the trust. 

I am uncertain about the specific structure and responsibilities of the trust council at Alexander Thamm GmbH, as this information may vary depending on the company's specific trust arrangements.

### Task 1.3 (a bit more advanced): How can you make the generation above retrieval augmented? Think of a super simple and manual way.

In [9]:
reference_text = """
Responsibility and Integration: The Trust Council (TC) will be able to make real changes at AT. It will be part of the Strategy Circle and will be involved in the processes of Appraisal, BSC (Variable Salary KPIs), and Work Contracts. This will help the TC focus on the things that matter most to you. 
If you're in a tricky situation, the TC will be a safe and private place for you to go. You can reach out to any TC member if you're worried about causing problems, want a second opinion, or just don't feel comfortable dealing with the issue directly.
And as a (somehow) democratic institution of course the TC will regularly inform you, the representees, about its work.
"""

In [10]:
input_prompt = """

Explain the role of a trust council at the Alexander Thamm GmbH. If you don't know the answer, say that you don't know.

The role is explained in this reference: \
    
{reference}

"""

In [11]:
response = chat_generator.invoke(input_prompt.format(reference=reference_text))
display(Markdown(response.content))

The Trust Council at Alexander Thamm GmbH plays a crucial role in decision-making processes, providing a safe and private space for employees to address concerns, seek advice, and ensure that their voices are heard. The TC is involved in strategic planning, performance evaluation, and contract negotiations, helping to prioritize issues that are important to employees. Additionally, the TC operates as a democratic institution, regularly informing employees about its work and seeking input from representees.

### Task 1.4 (if you are bored): Checkout the qdrant function 'search_documents'. Can you use this function to implement a simple RAG system using around the functions above?

Hints:
- you need the get_embedder function, the Document class and the instantiate_qclient and search_documents functions
- embed your query
- use search_documents
- use the from_qdrant_scored_point method of the Document class to retrieve the content of the best document
- print it - easy :)

In [12]:
from rag_llm_applications.embed import get_embedder
from rag_llm_applications.model import Document
from rag_llm_applications.qdrant import instantiate_qclient, search_documents

embedder = get_embedder('text-embedding-ada-002')

# only run this once - you can't instantiate it again. Either comment out or delete the instantiation (del qdrant) to run again.
qdrant = instantiate_qclient("data/qdrant")

In [13]:
embedded_question = embedder.embed_query('What is the role of a trust council at the Alexander Thamm GmbH?')

top_10_documents = search_documents(
    qdrant, "documents", embedded_question
)

best_doc = Document.from_qdrant_scored_point(top_10_documents[0]).content

display(Markdown(Document.from_qdrant_scored_point(top_10_documents[0]).content))

Rules of Procedure Trust Council [at] 
 
At its meeting on November 16, 2023, the Trust Council (TC) of Alexander Thamm GmbH 
adopted the following rules of procedure by a majority vote of its members: 
§ 1 Purpose 
The rules of procedure were adopted to determine the working mode of the TC.  
§ 2 Tasks of the TC Chairman 
Unless otherwise agreed, the Chairperson of the TC manages the day-to-day business and 
implements resolutions. 
This includes, for example 
- the preparation of meetings 
- granting and withdrawing the right to speak at Trust Council meetings 
- correspondence with the management, legal advisors and other parties 
- representing the Trust Council externally 
The above areas of responsibility can also be delegated to other members of the TC. 
§ 3 Representation of the TC Chairperson 
1 In the event that the TC Chairperson is unable to perform his/her duties, the Deputy TC 
Chairperson shall perform his/her duties. 
2. if it is not possible for the TC Chairperson to perform his/her duties due to his/her being 
prevented from doing so, he/she shall be replaced in the following order: in descending 
order of seniority. 
§ 4 Members of the Trust Council 
Rights of the members:  
1. in the course of dealing with personal complaints or personnel matters, the employees 
concerned should always be consulted. The hearing can also be carried out by individual 
members of the TC. 
Should a member resign from the TC during the term of office, the candidate from the same 
department with the most votes shall succeed him/her. If no candidate from the same 
department is available, the candidate with the highest total number of votes shall succeed. 
§ 5 Work planning of the TC 
1 The TC members are assigned special tasks. This includes, among other things, carrying out 
preparatory work for the discussion of the TC and the practical implementation of TC 
resolutions. 
2. with the exception of the tasks mentioned in § 2, neither the individual TC member nor 
the TC Chairperson or his/her deputy may make independent decisions. 
§ 6 TC meeting 
1. the TC meetings shall take place regularly and must be announced 7 days in advance.  
2. at least one meeting must be scheduled per quarter. 
3. it is also possible to convene an extraordinary meeting of the TC at any time if the TC 
chairperson or, if he/she is unable to do so, a deputy chairperson deems it necessary. 
4. virtual TC meetings may be held.

In [14]:
response = chat_generator.invoke(input_prompt.format(reference=best_doc))
display(Markdown(response.content))

The Trust Council at Alexander Thamm GmbH plays a crucial role in managing the day-to-day business and implementing resolutions. The Chairperson of the TC is responsible for tasks such as preparing meetings, corresponding with management and legal advisors, and representing the Trust Council externally. In the absence of the Chairperson, the Deputy TC Chairperson steps in to perform their duties. The members of the Trust Council are assigned specific tasks and are required to carry out preparatory work for discussions and the practical implementation of resolutions. Regular TC meetings are held, with at least one meeting scheduled per quarter, and virtual meetings may also be held if necessary. If a member resigns during their term, a successor is chosen based on departmental or overall votes.